In [1]:
!pip install pyspark

In [3]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types

In [6]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("TaxiHomework") \
    .master("local[*]") \
    .getOrCreate()

In [7]:
spark.version

'4.0.2'

In [11]:
ls -lh yellow_tripdata_2025-11.parquet

-rw-r--r-- 1 root root 68M Mar  5 18:58 yellow_tripdata_2025-11.parquet


In [12]:
df = spark.read.parquet("yellow_tripdata_2025-11.parquet")
df.printSchema()
df.show(5)

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)

+--------+--------------------+---------------------+---------------+------

In [13]:
df_repartitioned = df.repartition(4)

In [14]:
df_repartitioned.write.parquet("yellow_tripdata_2025_11_repartitioned")

In [16]:
!du -sh yellow_tripdata_2025_11_repartitioned/*

26M	yellow_tripdata_2025_11_repartitioned/part-00000-7190c697-35c0-45ce-bca5-ba17ed59b023-c000.snappy.parquet
26M	yellow_tripdata_2025_11_repartitioned/part-00001-7190c697-35c0-45ce-bca5-ba17ed59b023-c000.snappy.parquet
26M	yellow_tripdata_2025_11_repartitioned/part-00002-7190c697-35c0-45ce-bca5-ba17ed59b023-c000.snappy.parquet
26M	yellow_tripdata_2025_11_repartitioned/part-00003-7190c697-35c0-45ce-bca5-ba17ed59b023-c000.snappy.parquet
0	yellow_tripdata_2025_11_repartitioned/_SUCCESS


In [17]:
import os

folder = "yellow_tripdata_2025_11_repartitioned"

sizes = []

for file in os.listdir(folder):
    if file.endswith(".parquet"):
        size = os.path.getsize(os.path.join(folder, file)) / (1024*1024)
        sizes.append(size)
        print(file, round(size,2), "MB")

print("Average size:", round(sum(sizes)/len(sizes),2), "MB")

part-00003-7190c697-35c0-45ce-bca5-ba17ed59b023-c000.snappy.parquet 25.34 MB
part-00002-7190c697-35c0-45ce-bca5-ba17ed59b023-c000.snappy.parquet 25.33 MB
part-00000-7190c697-35c0-45ce-bca5-ba17ed59b023-c000.snappy.parquet 25.35 MB
part-00001-7190c697-35c0-45ce-bca5-ba17ed59b023-c000.snappy.parquet 25.33 MB
Average size: 25.34 MB


In [22]:
from pyspark.sql.functions import col, to_date

dataframe_for_15_november = df.filter(
    to_date(col("tpep_pickup_datetime")) == "2025-11-15"
)

dataframe_for_15_november.count()

162604

In [23]:
from pyspark.sql.functions import unix_timestamp, max

df_duration = df.withColumn(
    "trip_hours",
    (unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 3600
)

df_duration.select(max("trip_hours")).show()

+-----------------+
|  max(trip_hours)|
+-----------------+
|90.64666666666666|
+-----------------+



In [24]:
spark.sparkContext

<SparkContext master=local[*] appName=TaxiHomework>

In [25]:
spark.sparkContext.uiWebUrl

'http://15821e678edb:4040'

In [26]:
zones = spark.read.csv(
    "taxi_zone_lookup.csv",
    header=True,
    inferSchema=True
)

In [27]:
df_join = df.join(
    zones,
    df.PULocationID == zones.LocationID
)

In [28]:
from pyspark.sql.functions import count

df_join.groupBy("Zone") \
    .agg(count("*").alias("trip_count")) \
    .orderBy("trip_count") \
    .show()

+--------------------+----------+
|                Zone|trip_count|
+--------------------+----------+
|Governor's Island...|         1|
|Eltingville/Annad...|         1|
|       Arden Heights|         1|
|       Port Richmond|         3|
|       Rikers Island|         4|
|   Rossville/Woodrow|         4|
|         Great Kills|         4|
| Green-Wood Cemetery|         4|
|         Jamaica Bay|         5|
|         Westerleigh|        12|
|New Dorp/Midland ...|        14|
|             Oakwood|        14|
|        Crotona Park|        14|
|       West Brighton|        14|
|       Willets Point|        15|
|Breezy Point/Fort...|        16|
|Saint George/New ...|        17|
|       Broad Channel|        18|
|     Mariners Harbor|        21|
|Heartland Village...|        22|
+--------------------+----------+
only showing top 20 rows
